<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/02_construction_data_preparation_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# การเตรียมข้อมูลรายการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูล e-GP ปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 จากไฟล์ต้นทาง 8 ไฟล์

Notebook นี้รวมไฟล์ สกัดรายการ `จ้างก่อสร้าง` และตรวจโครงสร้างข้อมูลก่อนนำไปสำรวจต่อ โดยเก็บข้อมูลทุกวิธีจัดซื้อและทุกวงเงิน


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd

# ดาวน์โหลดฟอนต์ TH Sarabun New
!wget -q https://github.com/Phonbopit/sarabun-webfont/raw/master/fonts/thsarabunnew-webfont.ttf

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# เพิ่มฟอนต์ให้ Matplotlib
fm.fontManager.addfont(
    'thsarabunnew-webfont.ttf'
)

# กำหนดฟอนต์เริ่มต้นสำหรับ Matplotlib และ Seaborn
mpl.rc(
    'font',
    family='TH Sarabun New'
)
mpl.rcParams['axes.unicode_minus'] = False

sns.set_theme(
    style='whitegrid',
    font='TH Sarabun New'
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

base_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract/2569'
)

processed_dir = base_dir.parent / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(base_dir.glob('*.csv'))
construction_path = processed_dir / 'construction_contracts_2569.csv'

project_directory = base_dir.parents[3]
figure_directory = project_directory / 'figure'
figure_directory.mkdir(parents=True, exist_ok=True)

print(f'CSV files found: {len(csv_files)}')
print(f'Construction output: {construction_path}')
print(f'Figure directory: {figure_directory}')


## 1. ตรวจโครงสร้างไฟล์ต้นทาง

อ่านตัวอย่างจากไฟล์แรกเพื่อตรวจชื่อคอลัมน์และรูปแบบข้อมูลก่อนประมวลผลทุกไฟล์


In [ ]:
sample_data = pd.read_csv(
    csv_files[0],
    nrows=5
)

print(f'Sample file: {csv_files[0].name}')
print(f'Number of columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for position, column in enumerate(
    sample_data.columns,
    start=1
):
    print(f'{position:02d}. {column}')

## 2. รวมไฟล์และสกัดรายการจ้างก่อสร้าง

อ่านข้อมูลทีละไฟล์ นับจำนวนรายการตามประเภทโครงการ และเก็บเฉพาะรายการจ้างก่อสร้าง


In [ ]:
project_type_column = 'ชื่อประเภทโครงการ'

type_counts_list = []
construction_parts = []
file_summaries = []

for file_path in csv_files:
    data = pd.read_csv(file_path, low_memory=False)
    data.columns = data.columns.str.strip()

    project_type = (
        data[project_type_column]
        .astype('string')
        .str.strip()
        .fillna('ไม่ระบุ')
    )

    type_counts_list.append(project_type.value_counts())

    construction = data[project_type == 'จ้างก่อสร้าง'].copy()
    construction['source_file'] = file_path.name
    construction_parts.append(construction)

    file_summaries.append({
        'file_name': file_path.name,
        'total_rows': len(data),
        'construction_rows': len(construction)
    })

    print(
        f'{file_path.name}: {len(data):,} rows | '
        f'{len(construction):,} construction rows'
    )

construction_data = pd.concat(
    construction_parts,
    ignore_index=True
)

construction_data.to_csv(
    construction_path,
    index=False,
    encoding='utf-8-sig'
)

processing_summary = pd.DataFrame(file_summaries)

project_type_counts = (
    pd.concat(type_counts_list, axis=1)
    .fillna(0)
    .sum(axis=1)
    .astype('int64')
    .sort_values(ascending=False)
    .rename_axis(project_type_column)
    .reset_index(name='record_count')
)

total_records = processing_summary['total_rows'].sum()
total_construction_records = len(construction_data)
construction_pct = total_construction_records / total_records * 100

display(processing_summary)

print(f'Total records: {total_records:,}')
print(f'Construction records: {total_construction_records:,}')
print(f'Construction share: {construction_pct:.2f}%')
print(f'Output file: {construction_path}')


In [ ]:
# ตรวจยืนยันผลการประมวลผลกับข้อมูลชุดที่ใช้ในโครงการ
expected_file_count = 8
expected_total_records = 3_964_924
expected_construction_records = 180_079

assert len(csv_files) == expected_file_count, (
    f'Expected {expected_file_count} source files, '
    f'but found {len(csv_files)}'
)
assert total_records == expected_total_records, (
    f'Expected {expected_total_records:,} source rows, '
    f'but found {total_records:,}'
)
assert total_construction_records == expected_construction_records, (
    f'Expected {expected_construction_records:,} construction rows, '
    f'but found {total_construction_records:,}'
)
assert construction_path.exists(), (
    f'Output file was not created: {construction_path}'
)

print('Validation passed:')
print(f'- Source files: {len(csv_files)}')
print(f'- Source rows: {total_records:,}')
print(f'- Construction rows: {total_construction_records:,}')
print(f'- Output file exists: {construction_path.exists()}')

### สัดส่วนรายการตามประเภทโครงการ

เปรียบเทียบจำนวนรายการของแต่ละประเภท เพื่อดูว่างานจ้างก่อสร้างมีขนาดเท่าใดในข้อมูลทั้งหมด


In [ ]:
project_type_counts['record_pct'] = (
    project_type_counts['record_count']
    .div(
        project_type_counts[
            'record_count'
        ].sum()
    )
    .mul(100)
)

display(project_type_counts)

In [ ]:
major_types = (
    project_type_counts
    .head(4)
    .sort_values('record_pct', ascending=True)
    .copy()
)

bar_colors = [
    '#4C6A85'
    if project_type == 'จ้างก่อสร้าง'
    else '#A7B1BB'
    for project_type
    in major_types['ชื่อประเภทโครงการ']
]

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.barh(
    major_types['ชื่อประเภทโครงการ'],
    major_types['record_pct'],
    color=bar_colors,
    height=0.62
)

ax.bar_label(
    bars,
    labels=[
        f'{count:,.0f} รายการ ({percentage:.2f}%)'
        for count, percentage in zip(
            major_types['record_count'],
            major_types['record_pct']
        )
    ],
    padding=5,
    fontsize=12,
    color='#2F3437'
)

ax.set_title(
    f'งานจ้างก่อสร้างคิดเป็น {construction_pct:.2f}% '
    'ของรายการจัดซื้อจัดจ้าง',
    loc='left',
    fontsize=18,
    fontweight='bold',
    color='#2F3437',
    pad=18
)
ax.set_xlabel('สัดส่วนของจำนวนรายการ (%)', fontsize=12)
ax.set_ylabel('')
ax.set_xlim(0, major_types['record_pct'].max() * 1.24)
ax.tick_params(axis='both', colors='#555D63', labelsize=11)
ax.grid(axis='x', color='#E5E9ED', linewidth=0.8)
ax.grid(axis='y', visible=False)
ax.spines[:].set_visible(False)

ax.text(
    0,
    -0.18,
    f'ฐาน: {total_records:,.0f} รายการ | '
    'ข้อมูลสะสมถึง 30 กรกฎาคม 2569',
    transform=ax.transAxes,
    fontsize=11,
    color='#687178'
)

plt.tight_layout()

png_path = figure_directory / 'fig02_01_procurement_to_construction.png'
svg_path = figure_directory / 'fig02_01_procurement_to_construction.svg'
fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(svg_path, bbox_inches='tight', facecolor='white')

plt.show()
print(f'Figures saved: {png_path} and {svg_path}')


### ผลการสกัด

พบรายการจ้างก่อสร้าง 180,079 รายการ จากทั้งหมด 3,964,924 รายการ หรือ 4.54% โดยจำนวนนี้เป็นจำนวนแถวในไฟล์ต้นทาง ยังไม่ใช่จำนวนโครงการไม่ซ้ำ


## 3. ตรวจโครงสร้างข้อมูลจ้างก่อสร้าง

ตรวจจำนวนแถว รหัสโครงการ เลขที่สัญญา แถวซ้ำ และโครงการที่ปรากฏมากกว่าหนึ่งแถว


In [ ]:
construction_data = pd.read_csv(
    construction_path,
    low_memory=False
)

print(
    f'Shape: '
    f'{construction_data.shape}'
)

display(
    construction_data.head()
)

In [ ]:
validation_summary = pd.Series({
    'Rows': len(construction_data),
    'Columns': construction_data.shape[1],
    'Unique project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].nunique()
    ),
    'Unique contract number labels (not contract count)': (
        construction_data[
            'เลขที่สัญญา'
        ].nunique()
    ),
    'Missing project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].isna().sum()
    ),
    'Missing contract numbers': (
        construction_data[
            'เลขที่สัญญา'
        ].isna().sum()
    ),
    'Exact duplicate rows': (
        construction_data
        .drop(columns='source_file')
        .duplicated()
        .sum()
    )
})

display(
    validation_summary.to_frame(
        name='value'
    )
)

display(
    construction_data[
        'ชื่อประเภทโครงการ'
    ].value_counts(
        dropna=False
    )
)

In [ ]:
project_row_counts = (
    construction_data[
        'รหัสโครงการ'
    ]
    .value_counts()
)

print(
    'Projects with more than one row:',
    (project_row_counts > 1).sum()
)

print(
    'Maximum rows per project:',
    project_row_counts.max()
)

print(
    '\nMost frequent contract number labels '
    '(not a count of unique contracts):'
)

display(
    construction_data[
        'เลขที่สัญญา'
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)

repeated_project_ids = (
    project_row_counts.loc[
        project_row_counts > 1
    ]
    .head(3)
    .index
)

repeated_project_sample = (
    construction_data.loc[
        construction_data[
            'รหัสโครงการ'
        ].isin(
            repeated_project_ids
        ),
        [
            'รหัสโครงการ',
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            'ชื่อผู้ชนะการเสนอราคา',
            'เลขที่สัญญา',
            'วงเงินงบประมาณในสัญญา (บาท)'
        ]
    ]
    .sort_values(
        [
            'รหัสโครงการ',
            'เลขที่สัญญา'
        ]
    )
    .head(20)
)

display(repeated_project_sample)

### ความพร้อมของฟิลด์ที่ใช้วิเคราะห์

ตรวจความครบถ้วนของจังหวัด หน่วยงานย่อย ผู้รับจ้าง และวันที่เกิดรายการในระดับโครงการ


In [ ]:
project_id_column = 'รหัสโครงการ'
province_column = 'จังหวัด'
subagency_column = 'ชื่อหน่วยงานย่อย'
supplier_id_column = 'เลขประจำตัวนิติบุคคล 13 หลัก'
supplier_name_column = 'ชื่อผู้ชนะการเสนอราคา'
transaction_date_column = 'วันที่เกิดรายการ'

quality_columns = [
    province_column,
    subagency_column,
    supplier_id_column,
    supplier_name_column,
    transaction_date_column
]

for column in quality_columns:
    construction_data[column] = (
        construction_data[column]
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )

project_data_for_quality = (
    construction_data
    .drop_duplicates(subset=project_id_column, keep='first')
    .copy()
)

province_count_by_project = (
    construction_data
    .groupby(project_id_column, dropna=False)[province_column]
    .nunique(dropna=True)
)

project_supplier_pairs = (
    construction_data[
        [
            project_id_column,
            supplier_id_column,
            supplier_name_column
        ]
    ]
    .drop_duplicates()
)

data_quality_summary = pd.Series({
    'โครงการทั้งหมด': project_data_for_quality[project_id_column].nunique(),
    'โครงการที่จังหวัดหาย': project_data_for_quality[province_column].isna().sum(),
    'โครงการที่หน่วยงานย่อยหาย': project_data_for_quality[subagency_column].isna().sum(),
    'โครงการที่รหัสผู้รับจ้างหาย': project_data_for_quality[supplier_id_column].isna().sum(),
    'โครงการที่ชื่อผู้รับจ้างหาย': project_data_for_quality[supplier_name_column].isna().sum(),
    'โครงการที่วันที่เกิดรายการหาย': project_data_for_quality[transaction_date_column].isna().sum(),
    'โครงการที่พบมากกว่า 1 จังหวัด': province_count_by_project.gt(1).sum(),
    'คู่โครงการ–ผู้รับจ้างไม่ซ้ำ': len(
        project_supplier_pairs[
            [project_id_column, supplier_id_column]
        ].drop_duplicates()
    )
}, name='value')

display(data_quality_summary.to_frame())

province_spelling_check = (
    construction_data[province_column]
    .value_counts(dropna=False)
    .rename_axis(province_column)
    .reset_index(name='record_count')
)

display(province_spelling_check)


ค่าที่หายและโครงการที่มีจังหวัดมากกว่าหนึ่งค่าต้องตรวจสอบก่อนสร้างข้อมูลระดับโครงการ เพื่อไม่ให้การเลือกแถวแรกเปลี่ยนผลวิเคราะห์


## ไฟล์ผลลัพธ์

ไฟล์ `construction_contracts_2569.csv` มีรายการจ้างก่อสร้าง 180,079 แถว และ 178,978 โครงการไม่ซ้ำ พร้อมใช้สร้างข้อมูลระดับโครงการและสำรวจต่อใน Notebook 03
